In [ ]:
# Importing Modules and Data

folder1 = '1_test_sigma_0.3/'
folder2 = '2_test'
folder2a = '2a_test'
folder7 = '7_test_sigma_0.3'
folder6 = '6_test'
folder25 = '2e_test'
folder8 = '8_test_sigma_10'
folder85 = '8e_test'
folder9 = '9_test_sigma_0.3_r'
folder10 = '10_test_sigma_10_r'
folder11 = '11_test_sigma_0.3'
folder12 = '12_test_sigma_10'

folder = folder25
timesteps = [53] # [51]


# Edit Above

import os
import sys
import numpy as np
import scipy as sp

sys.path.append("../../")

import tristanVis.isolde as isolde
import tristanVis.tristanVis as tVis
import tristanVis.snippets as trS

#try:
#    trS.loadCustomStyles(style = 'default', fs = 15)
#except ValueError:
#    print("Value Error: Should be nothing to worry about.")


import matplotlib.pyplot as plt

sys.path.append("../")

from useful_funcs import *

In [ ]:
# Read Input

input_data = readInput(f"../../{folder}/input.2d_rec")

de = input_data['plasma:c_omp']
spec_nx = input_data['output:spec_nx']
spec_ny = input_data['output:spec_ny']
mx0 = input_data['grid:mx0']
my0 = input_data['grid:my0']
sigma = input_data['plasma:sigma']

xratio = mx0 / spec_nx / de
yratio = my0 / spec_ny / de

In [ ]:
# Script Inputs



#xrange = np.array([100, 140])
#yrange = np.array([420, 460])

#xrange = np.array([180, 225])
yrange = np.array([420, 460])

xrange = np.array([55, 65])
yrange = np.array([112.5, 122.5])

#xrange = None
#yrange = None

In [ ]:
species = 2
mass = 1

if (xrange is None):
    xbrange = np.array([0, spec_nx])

if (yrange is None):
    ybrange = np.array([0, spec_ny])

if (yrange is not None and yrange is not None):
    xbrange = np.floor(xrange / xratio).astype(int) + np.array([0, 1])
    ybrange = np.floor(yrange / yratio).astype(int) + np.array([0, 1])

    xbrange[0] = max(xbrange[0], 0)
    xbrange[1] = min(xbrange[1], spec_nx)

    ybrange[0] = max(ybrange[0], 0)
    ybrange[1] = min(ybrange[1], spec_ny)

xarange = xbrange * xratio
yarange = ybrange * yratio

if ((xrange is not None) or (yrange is not None)):
    print(f"Requested: [{xrange[0]:0.0f}, {xrange[1]:0.0f}] X [{yrange[0]:0.0f}, {yrange[1]:0.0f}]")
    print(f"Actual   : [{xarange[0]:0.0f}, {xarange[1]:0.0f}] X [{yarange[0]:0.0f}, {yarange[1]:0.0f}]")
    print(f"{(xbrange[1] - xbrange[0]) * (xbrange[1] - xbrange[0])} Cells")

In [ ]:
%matplotlib widget

plt.rcParams['text.usetex'] = True
plt.rcParams['font.family'] = 'serif'

plt.figure()

sigma_corr = False
mass_corr = False

for timestep in timesteps: 
    filename = f'../../{folder}/output/spec/' + 'spec.tot.%05d' % timestep

    print(filename)

    spectra = isolde.getSpectra(filename)

    spec_bins = spectra.ebins
    n = len(spec_bins)

    bin_sizes = np.diff(np.insert(spec_bins, 0, 0))

    spec_counts = np.zeros(n)

    for xbin in range(xbrange[0], xbrange[1]):
        for ybin in range(ybrange[0], ybrange[1]):
            spec_counts = np.add(spec_counts, spectra.getBySpatialBin(species, (xbin, ybin, 0))[2])

    corr_factor = 1

    if (sigma_corr):
        corr_factor /= sigma
    if (mass_corr):
        corr_factor *= mass

    plt.loglog(spec_bins * corr_factor, spec_counts / bin_sizes, markersize = 1.5, label = f"time = {timestep}")

plt.title(f"Species \# {species}, {np.sum(spec_counts):.2E} Particles \n x:[{xarange[0]:0.0f}, {xarange[1]:0.0f}] to y:[{yarange[0]:0.0f}, {yarange[1]:0.0f}]")
plt.legend(prop = {'size': 8})

if (sigma_corr and mass_corr):
    plt.xlabel("$(\gamma - 1) / \sigma (m / m_e)$")
if (sigma_corr and not mass_corr):
    plt.xlabel("$(\gamma - 1) / \sigma$")
if (not sigma_corr and mass_corr):
    plt.xlabel("$(\gamma - 1) * (m / m_e)$")
if (not sigma_corr and not mass_corr):
    plt.xlabel("$\gamma - 1$")



plt.ylabel("$dN/d \gamma$")

plt.grid()
plt.show()

In [ ]:
print(sigma)

In [ ]:
print(spectra.ebins)